In [ ]:
!pip install sqlalchemy psycopg2-binary python-dotenv pandas

In [2]:
import os
import pandas as pd
from dotenv import load_dotenv
from sqlalchemy import create_engine, text

# Carrega as credenciais do .env
load_dotenv()

DB_HOST = os.getenv("DB_HOST")
DB_PORT = os.getenv("DB_PORT", "5432")
DB_NAME = os.getenv("DB_NAME")
DB_USER = os.getenv("DB_USER")
DB_PASSWORD = os.getenv("DB_PASSWORD")

# String de conexão SQLAlchemy para PostgreSQL
DATABASE_URL = f"postgresql+psycopg2://{DB_USER}:{DB_PASSWORD}@{DB_HOST}:{DB_PORT}/{DB_NAME}?sslmode=require"

# Criar a engine
engine = create_engine(DATABASE_URL)

# Teste de validação
try:
    with engine.connect() as connection:
        result = connection.execute(text("SELECT version();"))
        print("Conexão realizada com sucesso!")
        print("Versão do Banco:", result.scalar())
except Exception as e:
    print("Erro ao conectar:", e)

Conexão realizada com sucesso!
Versão do Banco: PostgreSQL 17.10 (4f20678) on aarch64-unknown-linux-gnu, compiled by gcc (Debian 12.2.0-14+deb12u1) 12.2.0, 64-bit


In [3]:
query_tabelas = """
SELECT table_name 
FROM information_schema.tables 
WHERE table_schema = 'olist';
"""

df_tabelas = pd.read_sql(query_tabelas, engine)
df_tabelas

,table_name
0,olist_customers
1,olist_geolocation
2,olist_sellers
3,product_category_name_translation
4,olist_products
5,olist_order_items
6,olist_order_payments
7,olist_order_reviews
8,olist_orders


In [4]:
mapa_pedidos = {
    'order_id': 'ID do Pedido',
    'customer_id': 'ID Cliente (Pedido)',
    'order_status': 'Status do Pedido',
    'order_purchase_timestamp': 'Data da Compra',
    'order_approved_at': 'Data de Aprovação',
    'order_delivered_carrier_date': 'Data de Envio',
    'order_delivered_customer_date': 'Data da Entrega',
    'order_estimated_delivery_date': 'Data Estimada de Entrega'
}

mapa_vendas = {
    'order_id': 'ID do Pedido',
    'order_item_id': 'Sequencial do Item',
    'product_id': 'ID do Produto',
    'seller_id': 'ID do Vendedor',
    'shipping_limit_date': 'Data Limite de Envio',
    'price': 'Valor do Item',
    'freight_value': 'Valor do Frete'
}

mapa_clientes = {
    'customer_id': 'ID Cliente (Pedido)',
    'customer_unique_id': 'ID Único do Cliente',
    'customer_zip_code_prefix': 'CEP do Cliente',
    'customer_city': 'Cidade do Cliente',
    'customer_state': 'UF do Cliente'
}

mapa_produtos = {
    'product_id': 'Id do Produto',
    'product_category_name': 'Categoria do Produto',
    'product_photos_qty': 'Quantidade de Fotos',
    'product_weight_g': 'Peso (g)',
    'product_length_cm': 'Comprimento (cm)',
    'product_height_cm': 'Altura (cm)',
    'product_width_cm': 'Largura (cm)'
}

mapa_vendedores = {
    'seller_id': 'ID do Vendedor',
    'seller_zip_code_prefix': 'CEP do Vendedor',
    'seller_city': 'Cidade do Vendedor',
    'seller_state': 'UF do Vendedor'
}

mapa_pagamentos = {
    'order_id': 'ID do Pedido',
    'payment_sequential': 'Sequencial do Pagamento',
    'payment_type': 'Forma de Pagamento',
    'payment_installments': 'Quantidade de Parcelas',
    'payment_value': 'Valor Pago'
}

mapa_avaliacoes = {
    'review_id': 'ID da Avaliação',
    'order_id': 'ID do Pedido',
    'review_score': 'Nota da Avaliação',
    'review_comment_title': 'Título do Comentário',
    'review_comment_message': 'Mensagem do Comentário',
    'review_creation_date': 'Data de Criação da Avaliação',
    'review_answer_timestamp': 'Data de Resposta da Avaliação'
}

mapa_geolocalizacao = {
    'geolocation_zip_code_prefix': 'CEP Geolocalização',
    'geolocation_lat': 'Latitude',
    'geolocation_lng': 'Longitude',
    'geolocation_city': 'Cidade Geolocalização',
    'geolocation_state': 'UF Geolocalização'
}

mapa_traducao_categorias = {
    'product_category_name': 'Categoria do Produto',
    'product_category_name_english': 'Categoria do Produto (Inglês)'
}

with engine.connect() as conn:
    df_pedidos = pd.read_sql(text("SELECT * FROM olist.olist_orders;"), conn).rename(columns=mapa_pedidos)
    
    df_vendas = pd.read_sql(text("SELECT * FROM olist.olist_order_items;"), conn).rename(columns=mapa_vendas)
    
    df_clientes = pd.read_sql(text("SELECT * FROM olist.olist_customers;"), conn).rename(columns=mapa_clientes)

    df_produtos = pd.read_sql(text("SELECT * FROM olist.olist_products;"), conn).rename(columns=mapa_produtos)
    
    df_vendedores = pd.read_sql(text("SELECT * FROM olist.olist_sellers;"), conn).rename(columns=mapa_vendedores)

    df_pagamentos = pd.read_sql(text("SELECT * FROM olist.olist_order_payments;"), conn).rename(columns=mapa_pagamentos)

    df_avaliacoes = pd.read_sql(text("SELECT * FROM olist.olist_order_reviews;"), conn).rename(columns=mapa_avaliacoes)

    df_geolocalizacao = pd.read_sql(text("SELECT * FROM olist.olist_geolocation;"), conn).rename(columns=mapa_geolocalizacao)

    df_traducao_categorias = pd.read_sql(text("SELECT * FROM olist.product_category_name_translation;"), conn).rename(columns=mapa_traducao_categorias)

print("As 9 tabelas foram carregadas e renomeadas com sucesso!")

As 9 tabelas foram carregadas e renomeadas com sucesso!


In [5]:
dataframes = {
    "Pedidos": df_pedidos,
    "Vendas": df_vendas,
    "Clientes": df_clientes,
    "Produtos": df_produtos,
    "Vendedores": df_vendedores,
    "Pagamentos": df_pagamentos,
    "Avaliações": df_avaliacoes,
    "Geolocalização": df_geolocalizacao,
    "Tradução Categorias": df_traducao_categorias,
}

for nome, df in dataframes.items():
    print(f" {nome}: {df.shape[0]:,} linhas x {df.shape[1]} colunas".replace(',', '.'))

 Pedidos: 99.441 linhas x 8 colunas
 Vendas: 112.650 linhas x 7 colunas
 Clientes: 99.441 linhas x 5 colunas
 Produtos: 32.951 linhas x 9 colunas
 Vendedores: 3.095 linhas x 4 colunas
 Pagamentos: 103.886 linhas x 5 colunas
 Avaliações: 99.224 linhas x 7 colunas
 Geolocalização: 1.000.163 linhas x 5 colunas
 Tradução Categorias: 72 linhas x 2 colunas


In [6]:
df_pedidos['Data da Compra'] = pd.to_datetime(df_pedidos['Data da Compra'])

In [13]:
df_pedidos.head(2)

,ID do Pedido,ID Cliente (Pedido),Status do Pedido,Data da Compra,Data de Aprovação,Data de Envio,Data da Entrega,Data Estimada de Entrega
0,e481f51cbdc54678b7cc49136f2d6af7,9ef432eb6251297304e76186b10a928d,delivered,2017-10-02 10:56:33,2017-10-02 11:07:15,2017-10-04 19:55:00,2017-10-10 21:25:13,2017-10-18
1,53cdb2fc8bc7dce0b6741e2150273451,b0830fb4747a6c6d20dea0b8c802d7ef,delivered,2018-07-24 20:41:37,2018-07-26 03:24:27,2018-07-26 14:31:00,2018-08-07 15:27:45,2018-08-13


In [14]:
df_vendas.head(2)

,ID do Pedido,Sequencial do Item,ID do Produto,ID do Vendedor,Data Limite de Envio,Valor do Item,Valor do Frete
0,00010242fe8c5a6d1ba2dd792cb16214,1,4244733e06e7ecb4970a6e2683c13e61,48436dade18ac8b2bce089ec2a041202,2017-09-19 09:45:35,58.9,13.29
1,00018f77f2f0320c557190d7a144bdd3,1,e5f2d52b802189ee658865ca93d83a8f,dd7ddc04e1b6c2c614352b383efe2d36,2017-05-03 11:05:13,239.9,19.93


In [15]:
df_clientes.head(2)

,ID Cliente (Pedido),ID Único do Cliente,CEP do Cliente,Cidade do Cliente,UF do Cliente
0,06b8999e2fba1a1fbc88172c00ba8bc7,861eff4711a542e4b93843c6dd7febb0,14409,franca,SP
1,18955e83d337fd6b2def6b18a428ac77,290c77bc529b7ac935b93aa66c333dc3,09790,sao bernardo do campo,SP


In [16]:
df_produtos.head(2)

,Id do Produto,Categoria do Produto,product_name_lenght,product_description_lenght,Quantidade de Fotos,Peso (g),Comprimento (cm),Altura (cm),Largura (cm)
0,1e9e8ef04dbcff4541ed26657ea517e5,perfumaria,40.0,287.0,1.0,225.0,16.0,10.0,14.0
1,3aa071139cb16b67ca9e5dea641aaa2f,artes,44.0,276.0,1.0,1000.0,30.0,18.0,20.0


In [17]:
df_vendedores.head(2)

,ID do Vendedor,CEP do Vendedor,Cidade do Vendedor,UF do Vendedor
0,3442f8959a84dea7ee197c632cb2df15,13023,campinas,SP
1,d1b65fc7debc3361ea86b5f14c68d2e2,13844,mogi guacu,SP


In [18]:
df_pagamentos.head(2)

,ID do Pedido,Sequencial do Pagamento,Forma de Pagamento,Quantidade de Parcelas,Valor Pago
0,b81ef226f3fe1789b1e8b2acac839d17,1,credit_card,8,99.33
1,a9810da82917af2d9aefd1278f1dcfa0,1,credit_card,1,24.39


In [19]:
df_avaliacoes.head(2)

,ID da Avaliação,ID do Pedido,Nota da Avaliação,Título do Comentário,Mensagem do Comentário,Data de Criação da Avaliação,Data de Resposta da Avaliação
0,7bc2406110b926393aa56f80a40eba40,73fc7af87114b39712e6da79b0a377eb,4,None,None,2018-01-18,2018-01-18 21:46:59
1,80e641a11e56f04c1ad469d5645fdfde,a548910a1c6147796b98fdf73dbeba33,5,None,None,2018-03-10,2018-03-11 03:05:13


In [20]:
df_geolocalizacao.head(2)

,CEP Geolocalização,Latitude,Longitude,Cidade Geolocalização,UF Geolocalização
0,01037,-23.545621,-46.639292,sao paulo,SP
1,01046,-23.546081,-46.644820,sao paulo,SP
